In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder 
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, roc_auc_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, recall_score
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

df = pd.read_csv("Data\Data.csv")
print(f"   Shape: {df.shape}")

# Drop cluster (unsupervised artifact — leaks target info)
if "cluster" in df.columns:
    df.drop(columns=["cluster"], inplace=True)

# ─────────────────────────────────────────────
# 2. FEATURE DEFINITION
# ─────────────────────────────────────────────
numeric_features = [
    "serum_creatinine", "gfr", "bun", "serum_calcium",
    "c3_c4", "oxalate_levels", "urine_ph", "blood_pressure",
    "water_intake", "months"
]
binary_features = ["ana", "hematuria"]
categorical_features = [
    "physical_activity", "diet", "smoking", "alcohol",
    "painkiller_usage", "family_history", "weight_changes", "stress_level"
]
feature_cols = numeric_features + binary_features + categorical_features

# ─────────────────────────────────────────────
# 3. SHARED PREPROCESSOR
# ─────────────────────────────────────────────
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# تعديل هنا: استخدام OrdinalEncoder بدلاً من OneHotEncoder
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), 
])

# ملاحظة: تم فصل binary_features لتمريرها مع الميزات الرقمية أو الفئوية حسب رغبتك.
# هنا تركتها مع الرقمي ليتم عمل Standard Scaling لها، أو يمكنك نقلها مع الفئات.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     numeric_features + binary_features),
    ("cat", categorical_transformer, categorical_features),
], remainder="drop")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X = df[feature_cols]

# ═══════════════════════════════════════════════════════════
# TASK 2 — MULTI-CLASS: ckd_stage (0-5)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("TASK 2 — Multi-class: CKD Stage (0-5) [Using Ordinal Encoder]")
print("=" * 60)

y_stage = df["ckd_stage"]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X, y_stage, test_size=0.2, random_state=42, stratify=y_stage
)

models_stage = {
    "Logistic Regression": Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(
            class_weight="balanced", max_iter=2000,
            solver="lbfgs", random_state=42
        )),
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            max_depth=10, random_state=42, n_jobs=-1
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("pre", preprocessor),
        ("clf", GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05,
            max_depth=4, subsample=0.8, random_state=42
        )),
    ]),
    "Xgb": Pipeline([
       ("pre", preprocessor),
       ("xgb", XGBClassifier(random_state=42)) 
    ])
}

results_stage = {}
for name, pipe in models_stage.items():
    print(f"   > {name} ...", end=" ", flush=True)
    pipe.fit(X_tr2, y_tr2)
    y_pred = pipe.predict(X_te2)
    y_prob2 = pipe.predict_proba(X_te2)
    
    rec = recall_score(y_te2, y_pred, average="macro") 
    cv_acc = cross_val_score(pipe, X_tr2, y_tr2, cv=cv, scoring="accuracy", n_jobs=-1).mean()
    auc_ovr = roc_auc_score(y_te2, y_prob2, multi_class="ovr", average="macro")
    results_stage[name] = {"Recall": rec, "cv_acc": cv_acc, "auc_ovr": auc_ovr, "pipe": pipe}
    print(f"Recall={rec:.4f}  CV-Acc={cv_acc:.4f}  AUC-OvR={auc_ovr:.4f}")

best_stage = max(results_stage, key=lambda k: results_stage[k]["Recall"])
best_stage_pipe = results_stage[best_stage]["pipe"]
print(f"\n   Best model: {best_stage}")
print(classification_report(y_te2, best_stage_pipe.predict(X_te2),
                             target_names=[f"Stage {i}" for i in range(6)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"CKD Stage Classification — {best_stage}", fontsize=13, fontweight="bold")
cm2 = confusion_matrix(y_te2, best_stage_pipe.predict(X_te2))
ConfusionMatrixDisplay(cm2, display_labels=[f"S{i}" for i in range(6)]).plot(
    ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix (Stages 0-5)")

try:
    clf2 = best_stage_pipe[-1] 
    pre2 = best_stage_pipe.named_steps["pre"]
    
    # تعديل هنا: استخراج أسماء الميزات أصبح أسهل لأن OrdinalEncoder لا يغير الأسماء ولا ينشئ أعمدة جديدة
    feat_names2 = numeric_features + binary_features + categorical_features
    
    if hasattr(clf2, "feature_importances_"):
        imp2 = clf2.feature_importances_
    elif hasattr(clf2, "coef_"):
        imp2 = np.abs(clf2.coef_).mean(axis=0)
    else:
        raise AttributeError("Model does not have feature importances or coefficients.")
        
    top_idx2 = np.argsort(imp2)[-15:]
    axes[1].barh([feat_names2[i] for i in top_idx2], imp2[top_idx2], color="darkorange")
    axes[1].set_title("Top 15 Feature Importances")
    axes[1].set_xlabel("Importance")
except Exception as e:
    print(f"Could not plot feature importances: {e}")
    axes[1].set_visible(False)

plt.tight_layout()
plt.savefig("first_results.png", dpi=150)
plt.close()